# Unit 1 — Brisbane River surge (runnable notebook)

This notebook runs the **whole Unit 1 pipeline** end-to-end: build the bay,
solve the shallow-water equations forward to make synthetic tide-gauge data,
animate the surge, recover the unknown river source two ways (a Tikhonov
adjoint inverse and a naive PINN), draw the recovery plot, and finally run the
GPU version of the solver.

It is a convenience for hands-on experimentation — the course **site** (the
Quarto `unit_01.qmd`) carries the explanations; this notebook is just the code,
with light section headings. It is **not** served on the site.

**Each stage `include`s the matching `scripts/*.jl`.** Because those scripts
*write* their data and figures, the first cell finds the course copy of Unit 1
(even the read-only one on the hub, or if you've copied just this notebook into
your own folder), clones it into `~/unit_01_run`, and runs everything from there.
Use the **Julia 1.12** kernel.

> Run the cells **in order** — later stages read the CSVs/snapshots that earlier
> stages write.


In [ ]:
# This notebook runs the real scripts/*.jl, which WRITE into data/ and figures/.
#  1) Find the (possibly read-only) course copy of unit_01 — wherever Jupyter was
#     launched, INCLUDING when you've copied just this notebook into your own folder.
#  2) Clone it into a writable folder in your home and run everything from there.
function find_unit01()
    cands = ["scripts", "units/unit_01/scripts", "../scripts",
             joinpath(homedir(), "course-materials", "units", "unit_01", "scripts"),
             "/home/efs/_shared/course-materials/units/unit_01/scripts"]
    for c in cands
        isfile(joinpath(c, "build_bay.jl")) && return normpath(abspath(joinpath(c, "..")))
    end
    error("Could not find unit_01/scripts — set SRC by hand to your unit_01 folder.")
end
const SRC  = find_unit01()
const WORK = joinpath(homedir(), "unit_01_run")
isdir(WORK) || cp(SRC, WORK)
chmod(WORK, 0o755; recursive = true)          # the shared course copy may be read-only
const SCRIPTS = joinpath(WORK, "scripts")
@info "Unit 1: found course copy, running from a writable clone" SRC WORK

# Run each script in its OWN module, so its top-level `const`s (FIG_DIR, DATA_DIR,
# …) never collide with another stage's. The scripts pass data through files, not
# globals, so isolation is safe; we give each module a path-aware `include`.
function stage(name)
    println("\n", "="^70, "\n>>> ", name, "\n", "="^70)
    m = Module(:Stage)
    Core.eval(m, :(include(p) = Base.include($m, p)))
    Base.include(m, joinpath(SCRIPTS, name))
end


## 1. Build the bay
Hand-built bathymetry, land mask, gauge layout, and the bathymetry figure
(`figures/bathymetry.png`). Writes the CSVs in `data/`.


In [ ]:
stage("build_bay.jl")


## 2. Forward shallow-water solve
Linearised SWE on an Arakawa-C grid: propagate the prescribed river-mouth surge
across the bay, sample the four tide gauges, and save snapshots. (~12 s on a laptop CPU.)


In [ ]:
stage("generate_surge_data.jl")


## 3. Animate the surge
Per-snapshot PNG frames (`figures/surge_frames/`) and an optional GIF
(`figures/surge_animation.gif`).


In [ ]:
stage("generate_surge_frames.jl")


In [ ]:
# Optional: assemble the frames into a GIF movie.
stage("generate_surge_animation.jl")


## 4. Inverse problem — adjoint / Tikhonov
Use the simulator as a linear forward map, build the bay's Green's function,
and solve a smoothness-regularised (Tikhonov) least-squares problem for the
source ψ(t). No neural network. (~15 s.)


In [ ]:
stage("train_inverse_adjoint.jl")


## 5. Inverse problem — naive PINN
Train a small MLP η_θ(x,y,t) to match the gauges **and** satisfy the wave
equation at random collocation points; read the recovered source off at the
river-mouth cell. (~3 min on CPU. Needs Lux + Zygote, which are in `@pinn`.)


In [ ]:
stage("train_inverse_pinn.jl")


## 6. The recovery plot
The centrepiece: true ψ(t) vs the two recoveries, plus the four gauge fits
(`figures/inverse_recovery.png`). Both methods fit the gauges; only the adjoint
recovers the source — the ill-posedness made visible.


In [ ]:
stage("render_recovery_plot.jl")


## 7. The same solve, on a GPU
The identical linearised-SWE model written as whole-array broadcasts, so the
same code runs on CPU (`Array`) or GPU (`CuArray`). Refines the bay and races
the two.

> **GPU required.** This needs `CUDA.jl` and an NVIDIA GPU — run it on the GPU
> hub (or any CUDA machine). On a CPU-only box it will report that no GPU was
> found and just time the CPU path.


In [ ]:
stage("surge_gpu.jl")
